In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
torch.set_printoptions(precision=2, sci_mode=False)

In [4]:
def show(name, t):
    print(f"\n=== {name}   shape={tuple(t.shape)}   stride={t.stride()}   contiguous={t.is_contiguous()}")
    print(t)

### Step 0: Setup

In [5]:
# T = 5 tokens, C = 3, one sentence (B = 1)
X = torch.tensor([[1., 0., 1.],   # The
                  [0., 2., 1.],   # tall
                  [1., 1., 0.],   # man
                  [2., 0., 1.],   # runs
                  [0., 1., 2.]])  # .
X = X.unsqueeze(0)                # [B, T, C] = [1, 5, 3]
print(X)

tensor([[[1., 0., 1.],
         [0., 2., 1.],
         [1., 1., 0.],
         [2., 0., 1.],
         [0., 1., 2.]]])


In [6]:
B, T, C = X.shape
h = 2                 # number of heads
d_k = 2               # size of one head
hd_k = h * d_k        # attention_dim = 4

### Step 1: Weights

In [7]:
# C x hd_k = 3 x 4. Columns 0-1 = head 0, columns 2-3 = head 1.

#                    head 0  | head 1
W_Q = torch.tensor([[1., 0.,   1.,  0.],
                    [0., 1.,   0.,  0.],
                    [1., 1.,   0., -1.]])
W_K = torch.tensor([[0., 1.,   1.,  0.],
                    [1., 0.,   1.,  0.],
                    [1., 0.,  -1.,  1.]])
W_V = torch.tensor([[1., 0.,   2.,  0.],
                    [0., 1.,   0.,  1.],
                    [1., 0.,   0.,  2.]])

In [8]:
# hd_k x C = 4 x 3. Rows 0-1 = head 0's W^O_0, rows 2-3 = head 1's W^O_1.
W_O = torch.tensor([[1., 0., 1.],
                    [0., 1., 0.],
                    [1., 0., 0.],
                    [0., 1., 1.]])

### Step 2: project (all heads in one matmul)

In [20]:
Q = X @ W_Q                       # [B, T, hd_k]
K = X @ W_K
V = X @ W_V
show("Step 2: Q_f = X @ W_Q", Q)
show("Step 2: K_f = X @ W_K", K)
show("Step 2: V_f = X @ W_V", V)


=== Step 2: Q_f = X @ W_Q   shape=(1, 5, 4)   stride=(20, 4, 1)   contiguous=True
tensor([[[ 2.,  1.,  1., -1.],
         [ 1.,  3.,  0., -1.],
         [ 1.,  1.,  1.,  0.],
         [ 3.,  1.,  2., -1.],
         [ 2.,  3.,  0., -2.]]])

=== Step 2: K_f = X @ W_K   shape=(1, 5, 4)   stride=(20, 4, 1)   contiguous=True
tensor([[[ 1.,  1.,  0.,  1.],
         [ 3.,  0.,  1.,  1.],
         [ 1.,  1.,  2.,  0.],
         [ 1.,  2.,  1.,  1.],
         [ 3.,  0., -1.,  2.]]])

=== Step 2: V_f = X @ W_V   shape=(1, 5, 4)   stride=(20, 4, 1)   contiguous=True
tensor([[[2., 0., 2., 2.],
         [1., 2., 0., 4.],
         [1., 1., 2., 1.],
         [3., 0., 4., 2.],
         [2., 1., 0., 5.]]])


### Step 3: Split Heads

In [21]:
Q = Q.view(B, T, h, d_k)          # [B, T, h, d_k]   no copy
show("Step 3a: Q.view(B, T, h, d_k)", Q)
Q = Q.transpose(1, 2)             # [B, h, T, d_k]   no copy, strides swap
show("Step 3b: Q.transpose(1, 2)", Q)


=== Step 3a: Q.view(B, T, h, d_k)   shape=(1, 5, 2, 2)   stride=(20, 4, 2, 1)   contiguous=True
tensor([[[[ 2.,  1.],
          [ 1., -1.]],

         [[ 1.,  3.],
          [ 0., -1.]],

         [[ 1.,  1.],
          [ 1.,  0.]],

         [[ 3.,  1.],
          [ 2., -1.]],

         [[ 2.,  3.],
          [ 0., -2.]]]])

=== Step 3b: Q.transpose(1, 2)   shape=(1, 2, 5, 2)   stride=(20, 2, 4, 1)   contiguous=False
tensor([[[[ 2.,  1.],
          [ 1.,  3.],
          [ 1.,  1.],
          [ 3.,  1.],
          [ 2.,  3.]],

         [[ 1., -1.],
          [ 0., -1.],
          [ 1.,  0.],
          [ 2., -1.],
          [ 0., -2.]]]])


In [22]:
K = K.view(B, T, h, d_k).transpose(1, 2)
V = V.view(B, T, h, d_k).transpose(1, 2)
show("Step 3: K split", K)
show("Step 3: V split", V)


=== Step 3: K split   shape=(1, 2, 5, 2)   stride=(20, 2, 4, 1)   contiguous=False
tensor([[[[ 1.,  1.],
          [ 3.,  0.],
          [ 1.,  1.],
          [ 1.,  2.],
          [ 3.,  0.]],

         [[ 0.,  1.],
          [ 1.,  1.],
          [ 2.,  0.],
          [ 1.,  1.],
          [-1.,  2.]]]])

=== Step 3: V split   shape=(1, 2, 5, 2)   stride=(20, 2, 4, 1)   contiguous=False
tensor([[[[2., 0.],
          [1., 2.],
          [1., 1.],
          [3., 0.],
          [2., 1.]],

         [[2., 2.],
          [0., 4.],
          [2., 1.],
          [4., 2.],
          [0., 5.]]]])


In [23]:
# What goes wrong if you skip the transpose and view straight to [B, h, T, d_k]:
Q_wrong = (X @ W_Q).view(B, h, T, d_k)
show("WRONG: (X @ W_Q).view(B, h, T, d_k)  -- 'head 0' mixes tokens and heads", Q_wrong)


=== WRONG: (X @ W_Q).view(B, h, T, d_k)  -- 'head 0' mixes tokens and heads   shape=(1, 2, 5, 2)   stride=(20, 10, 2, 1)   contiguous=True
tensor([[[[ 2.,  1.],
          [ 1., -1.],
          [ 1.,  3.],
          [ 0., -1.],
          [ 1.,  1.]],

         [[ 1.,  0.],
          [ 3.,  1.],
          [ 2., -1.],
          [ 2.,  3.],
          [ 0., -2.]]]])


### Step 4: Scores

In [24]:
attention_matrix = Q @ K.transpose(-2, -1)          # [B, h, T, T]
attention_scores = attention_matrix / (d_k ** 0.5)
show("Step 4: attention_scores", attention_scores)


=== Step 4: attention_scores   shape=(1, 2, 5, 5)   stride=(50, 25, 5, 1)   contiguous=True
tensor([[[[ 2.12,  4.24,  2.12,  2.83,  4.24],
          [ 2.83,  2.12,  2.83,  4.95,  2.12],
          [ 1.41,  2.12,  1.41,  2.12,  2.12],
          [ 2.83,  6.36,  2.83,  3.54,  6.36],
          [ 3.54,  4.24,  3.54,  5.66,  4.24]],

         [[-0.71,  0.00,  1.41,  0.00, -2.12],
          [-0.71, -0.71,  0.00, -0.71, -1.41],
          [ 0.00,  0.71,  1.41,  0.71, -0.71],
          [-0.71,  0.71,  2.83,  0.71, -2.83],
          [-1.41, -1.41,  0.00, -1.41, -2.83]]]])


### Step 5: Causal Mask

In [25]:
mask = torch.tril(torch.ones(T, T))    # [T, T], broadcasts over [B, h]
mask = (mask == 0)
masked_scores = attention_scores.masked_fill(mask, float('-inf'))
show("Step 5: masked_scores", masked_scores)


=== Step 5: masked_scores   shape=(1, 2, 5, 5)   stride=(50, 25, 5, 1)   contiguous=True
tensor([[[[ 2.12,  -inf,  -inf,  -inf,  -inf],
          [ 2.83,  2.12,  -inf,  -inf,  -inf],
          [ 1.41,  2.12,  1.41,  -inf,  -inf],
          [ 2.83,  6.36,  2.83,  3.54,  -inf],
          [ 3.54,  4.24,  3.54,  5.66,  4.24]],

         [[-0.71,  -inf,  -inf,  -inf,  -inf],
          [-0.71, -0.71,  -inf,  -inf,  -inf],
          [ 0.00,  0.71,  1.41,  -inf,  -inf],
          [-0.71,  0.71,  2.83,  0.71,  -inf],
          [-1.41, -1.41,  0.00, -1.41, -2.83]]]])


### Step 6: Softmax, one A per head

In [26]:
A = F.softmax(masked_scores, dim=-1)                # [B, h, T, T]
show("Step 6: A", A)
print("row sums:", A.sum(dim=-1))


=== Step 6: A   shape=(1, 2, 5, 5)   stride=(50, 25, 5, 1)   contiguous=True
tensor([[[[1.00, 0.00, 0.00, 0.00, 0.00],
          [0.67, 0.33, 0.00, 0.00, 0.00],
          [0.25, 0.50, 0.25, 0.00, 0.00],
          [0.03, 0.89, 0.03, 0.05, 0.00],
          [0.07, 0.14, 0.07, 0.58, 0.14]],

         [[1.00, 0.00, 0.00, 0.00, 0.00],
          [0.50, 0.50, 0.00, 0.00, 0.00],
          [0.14, 0.28, 0.58, 0.00, 0.00],
          [0.02, 0.09, 0.79, 0.09, 0.00],
          [0.14, 0.14, 0.56, 0.14, 0.03]]]])
row sums: tensor([[[1.00, 1.00, 1.00, 1.00, 1.00],
         [1.00, 1.00, 1.00, 1.00, 1.00]]])


### Step 7: Blend values (multiply attention matrix with V)

In [27]:
Z = A @ V                                           # [B, h, T, d_k]
show("Step 7: Z = A @ V", Z)


=== Step 7: Z = A @ V   shape=(1, 2, 5, 2)   stride=(20, 10, 2, 1)   contiguous=True
tensor([[[[2.00, 0.00],
          [1.67, 0.66],
          [1.25, 1.26],
          [1.13, 1.82],
          [2.37, 0.49]],

         [[2.00, 2.00],
          [1.00, 3.00],
          [1.43, 1.99],
          [2.00, 1.40],
          [1.93, 1.81]]]])


### Step 8: Merge heads

In [28]:
Z_t = Z.transpose(1, 2)                             # [B, T, h, d_k]   no copy
show("Step 8a: Z.transpose(1, 2)", Z_t)


=== Step 8a: Z.transpose(1, 2)   shape=(1, 5, 2, 2)   stride=(20, 2, 10, 1)   contiguous=False
tensor([[[[2.00, 0.00],
          [2.00, 2.00]],

         [[1.67, 0.66],
          [1.00, 3.00]],

         [[1.25, 1.26],
          [1.43, 1.99]],

         [[1.13, 1.82],
          [2.00, 1.40]],

         [[2.37, 0.49],
          [1.93, 1.81]]]])


In [29]:
print("\nman's 4 numbers in memory (flat offsets):")
flat = Z.flatten()                                  # Z itself is contiguous, so this is its memory
for off in (4, 5, 14, 15):
    print(f"  offset {off:2d} -> {flat[off].item():.2f}")


man's 4 numbers in memory (flat offsets):
  offset  4 -> 1.25
  offset  5 -> 1.26
  offset 14 -> 1.43
  offset 15 -> 1.99


In [30]:
try:
    Z_t.view(B, T, hd_k)
except RuntimeError as e:
    print("\nStep 8b: Z_t.view(B, T, hd_k) FAILED:\n ", e)


Step 8b: Z_t.view(B, T, hd_k) FAILED:
  view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.


In [31]:
Z_c = Z_t.contiguous()              # copy into token-major order
show("Step 8c: Z_t.contiguous()", Z_c)
Z_merged = Z_c.view(B, T, hd_k)                     # [B, T, hd_k]   free now
show("Step 8d: .view(B, T, hd_k)", Z_merged)


=== Step 8c: Z_t.contiguous()   shape=(1, 5, 2, 2)   stride=(20, 4, 2, 1)   contiguous=True
tensor([[[[2.00, 0.00],
          [2.00, 2.00]],

         [[1.67, 0.66],
          [1.00, 3.00]],

         [[1.25, 1.26],
          [1.43, 1.99]],

         [[1.13, 1.82],
          [2.00, 1.40]],

         [[2.37, 0.49],
          [1.93, 1.81]]]])

=== Step 8d: .view(B, T, hd_k)   shape=(1, 5, 4)   stride=(20, 4, 1)   contiguous=True
tensor([[[2.00, 0.00, 2.00, 2.00],
         [1.67, 0.66, 1.00, 3.00],
         [1.25, 1.26, 1.43, 1.99],
         [1.13, 1.82, 2.00, 1.40],
         [2.37, 0.49, 1.93, 1.81]]])


### Step 9: output projection

In [32]:
out = Z_merged @ W_O                                # [B, T, C]
show("Step 9: out = Z_merged @ W_O", out)


=== Step 9: out = Z_merged @ W_O   shape=(1, 5, 3)   stride=(15, 3, 1)   contiguous=True
tensor([[[4.00, 2.00, 4.00],
         [2.67, 3.66, 4.67],
         [2.68, 3.25, 3.24],
         [3.13, 3.22, 2.53],
         [4.30, 2.30, 4.18]]])


### check if `out` is the sum of each head's own projection

In [33]:
# ----------------------------------------------------------------
head_parts = Z[:, 0] @ W_O[:2] + Z[:, 1] @ W_O[2:]
print("\nout == Z_0 @ W^O_0 + Z_1 @ W^O_1 ?", torch.allclose(out, head_parts))


out == Z_0 @ W^O_0 + Z_1 @ W^O_1 ? True


### head 0 reproduces SHA Part 2's single-head attention (same W_Q, W_K; Part 2's 3x4 W_V == our fused W_V)

In [34]:
Q2, K2, V2 = X[0] @ W_Q[:, :2], X[0] @ W_K[:, :2], X[0] @ W_V
S2 = (Q2 @ K2.T / d_k ** 0.5).masked_fill(mask, float('-inf'))
Z_part2 = F.softmax(S2, dim=-1) @ V2
print("head 0's A == Part 2's A ?", torch.allclose(A[0, 0], F.softmax(S2, dim=-1)))
print("Z_merged[:, :2] == Part 2's Z[:, :2] ?", torch.allclose(Z_merged[0, :, :2], Z_part2[:, :2]))


head 0's A == Part 2's A ? True
Z_merged[:, :2] == Part 2's Z[:, :2] ? True


### Bonus: h = 12, d_k = 64 case

In [43]:
print("\n\n=== Bonus: h = 12, d_k = 64, T = 5 (strides only)")
Z12 = torch.randn(1, 12, 5, 64)                     # A @ V output, [B, h, T, d_k]
Z12_t = Z12.transpose(1, 2)                         # [B, T, h, d_k]
print(f"A @ V           |{tuple(Z12.shape)} {Z12.stride()} \n")
print(f"transpose(1, 2) |{tuple(Z12_t.shape)} {Z12_t.stride()} contiguous = {Z12_t.is_contiguous()} \n")
try:
    Z12_t.view(1, 5, 768)
except RuntimeError as e:
    print(f"view(1, 5, 768) | FAILED: {e}\n")
Z12_c = Z12_t.contiguous()
print(f"contiguous()    |{tuple(Z12_c.shape)} {Z12_c.stride()}\n")
print(f"view(1, 5, 768) |{tuple(Z12_c.view(1, 5, 768).shape)} {Z12_c.view(1, 5, 768).stride()}\n")




=== Bonus: h = 12, d_k = 64, T = 5 (strides only)
A @ V           |(1, 12, 5, 64) (3840, 320, 64, 1) 

transpose(1, 2) |(1, 5, 12, 64) (3840, 64, 320, 1) contiguous = False 

view(1, 5, 768) | FAILED: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

contiguous()    |(1, 5, 12, 64) (3840, 768, 64, 1)

view(1, 5, 768) |(1, 5, 768) (3840, 768, 1)

